# EDBT dataset characterization — BRGM piezometers and forecasting benchmarks

**Purpose.** Mount Google Drive; locate the **17 piezometer CSVs used by your EDBT benchmark** (`piezo1`–`piezo9`, `piezo11`–`piezo18`); inspect their actual timestamps and covariates; calculate reproducible seasonality/trend/noise summaries and an ADF test; export CSVs and publication-ready LaTeX tables. This notebook **does not run the counterfactual methods or retrain the GRUs**.

**Important provenance.** The uploaded screenshot reports Weather as hourly, 21 variables, 52,696 observations. A commonly distributed Weather benchmark has 10-minute observations; **the CSV timestamps, not the screenshot, decide your eventual paper value.** The screenshot's qualitative high/medium/low categories have no supplied thresholds. This notebook transparently defines its *own* category cutoffs, which cannot be claimed to reproduce the screenshot's original assessments.

**Paper presentation:** one compact, five-row main-paper table (BRGM aggregate + four public datasets) and a 21-row full table for an appendix/supplement. If you also supply benchmark CSVs, this notebook computes their characteristics with the **same definitions** as BRGM; otherwise public entries are explicitly marked *reference values supplied in the screenshot* and are **not** treated as measured results.

In [21]:
# Run in Google Colab. Authorize access to the Google account that has the piezometer data.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

# EDIT THIS: folder containing the 17 piezometer CSV files (subfolders are okay).
DATA_ROOT = Path('/content/drive/MyDrive/data/piezos')

# Optional: a Drive folder containing ETTh1.csv, ETTh2.csv, weather.csv, electricity.csv.
# Set to None if you have not downloaded these yet.
PUBLIC_ROOT = None  # e.g. Path('/content/drive/MyDrive/data/forecasting')

OUTPUT_ROOT = Path('/content/drive/MyDrive/EDBT_dataset_characterization')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('BRGM folder:', DATA_ROOT, 'exists:', DATA_ROOT.is_dir())
print('Output folder:', OUTPUT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BRGM folder: /content/drive/MyDrive/Colab_Notebooks/Year1_PhD_Codes/03_Literature_Review_CE_for_TS_Forecasting/data/piezos exists: True
Output folder: /content/drive/MyDrive/EDBT_dataset_characterization


## Assumptions and configurable settings

The BRGM file names and forecast configuration below are read from your supplied `EDBT_unified_CF_benchmark_all_piezometers_with_horizon_validity(5).ipynb`. Its expected columns are `date`, `level`, `PRELIQ_Q`, `T_Q`, `ETP_Q`; its history is 336 steps, horizon 24, and training portion is the first 50% of each chronologically sorted series. The dataset characterization below uses the **training portion for distribution/ADF/STL fitting**, to avoid peeking at test data when reporting properties relevant to model selection. Dataset length, date range, and missingness describe the **complete** CSV.

Seasonality is defined as `max(0, 1 − Var(R) / Var(S + R))`, trend as `max(0, 1 − Var(R) / Var(T + R))`, using robust STL components (`T` trend, `S` seasonal, `R` residual). Scores are clipped to [0, 1]. Noise ratio is `Var(R) / Var(observed)`; this is a **descriptive proxy**, not an estimated physical measurement-error variance. Stationarity: ADF p-value on training *target*; `p < 0.05` rejects the unit-root null; failure to reject **does not prove nonstationarity**. These quantities are target-specific, whereas some literature tables do not specify how they aggregate across variables.

The category thresholds below are deliberately **our own explicit reporting conventions**, not thresholds inferred from the screenshot. Prefer quantitative scores in the paper wherever possible.

In [22]:
import sys, subprocess, importlib.util
if importlib.util.find_spec('statsmodels') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'statsmodels'])
import re, json, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

PIEZO_NAMES = [f'piezo{i}' for i in list(range(1,10)) + list(range(11,19))]
PUBLIC_NAMES = ['ETTh1','ETTh2','Weather','Electricity']
TARGETS = {'ETTh1':'OT', 'ETTh2':'OT', 'Weather':'T (degC)'}
DATE_ALIASES = ('date', 'datetime', 'timestamp', 'time')
TRAIN_RATIO = 0.50   # matches the uploaded CF notebook
INPUT_LEN, HORIZON = 336, 24
MAX_STL_TRAIN_POINTS = 6000  # contiguous end of train, for speed; see method note in table
MAX_ADF_TRAIN_POINTS = 5000
# Override season length (number of time steps per cycle), e.g. {'piezo7':365}
SEASONAL_PERIOD_OVERRIDE = {}
# Specify if a data source has no parseable timestamp; only use if definitely known.
FREQUENCY_OVERRIDE = {}  # e.g. {'piezo7':'1 day'}

def strength_label(value):
    if not np.isfinite(value): return 'n/a'
    if value >= 0.70: return 'High'
    if value >= 0.40: return 'Med.-high'
    if value >= 0.20: return 'Medium'
    return 'Low'

def noise_label(noise_ratio):
    if not np.isfinite(noise_ratio): return 'n/a'
    if noise_ratio < 0.10: return 'Low'
    if noise_ratio < 0.30: return 'Medium'
    return 'High'

print('Expected BRGM datasets:', len(PIEZO_NAMES), PIEZO_NAMES)


Expected BRGM datasets: 17 ['piezo1', 'piezo2', 'piezo3', 'piezo4', 'piezo5', 'piezo6', 'piezo7', 'piezo8', 'piezo9', 'piezo11', 'piezo12', 'piezo13', 'piezo14', 'piezo15', 'piezo16', 'piezo17', 'piezo18']


## Find files — inspect the list before calculating

Only CSVs with the **exact 17 configured piezometer stems** are used. This prevents accidentally counting output/summary CSVs in Drive. Folders are searched recursively; duplicate stems are treated as an error so a random older copy cannot be chosen silently.

In [23]:
def discover_exact_csvs(root, names):
    if root is None or not Path(root).is_dir(): return {}, list(names), {}
    candidates = list(Path(root).rglob('*.csv'))
    found, dup = {}, {}
    for name in names:
        matches = [p for p in candidates if p.stem.casefold() == name.casefold()]
        if len(matches) == 1: found[name] = matches[0]
        elif len(matches) > 1: dup[name] = [str(p) for p in matches]
    missing = [name for name in names if name not in found and name not in dup]
    return found, missing, dup

piezo_files, missing_piezos, duplicate_piezos = discover_exact_csvs(DATA_ROOT, PIEZO_NAMES)
public_files, missing_public, duplicate_public = discover_exact_csvs(PUBLIC_ROOT, PUBLIC_NAMES)
print('Found BRGM:', len(piezo_files), '/ 17')
display(pd.DataFrame([{'name': k, 'path': str(v)} for k,v in piezo_files.items()]))
print('Missing:', missing_piezos, '\nDuplicates:', duplicate_piezos)
print('Public CSVs found:', list(public_files))
if duplicate_piezos or duplicate_public:
    raise ValueError('Duplicate dataset CSV stems: resolve paths/duplicates before proceeding.')
if missing_piezos:
    print('WARNING: not all 17 BRGM files found. Check DATA_ROOT; no missing entries will be invented.')
if not piezo_files:
    raise FileNotFoundError('No piezometer CSVs found. Set DATA_ROOT to the folder used by your CF notebook.')


Found BRGM: 17 / 17


,name,path
0,piezo1,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
1,piezo2,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
2,piezo3,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
3,piezo4,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
4,piezo5,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
5,piezo6,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
6,piezo7,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
7,piezo8,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
8,piezo9,/content/drive/MyDrive/Colab_Notebooks/Year1_P...
9,piezo11,/content/drive/MyDrive/Colab_Notebooks/Year1_P...


Missing: [] 
Duplicates: {}
Public CSVs found: []


## Load, inspect timestamps, and estimate train-only time-series properties

No silent imputation of missing rows/targets. The decomposition uses a **contiguous nonmissing final segment of training data**; if the series contains gaps in calendar time or the target has missing values, STL/ADF are skipped and a quality flag is emitted, rather than assuming that consecutive rows mean consecutive time steps. For a short series, annual decomposition may not have enough cycles; the chosen period is exported for review.

In [24]:
def resolve_date_column(df):
    lookup = {c.casefold():c for c in df.columns}
    for candidate in DATE_ALIASES:
        if candidate in lookup: return lookup[candidate]
    return None

def find_frequency(times, override=None):
    if override is not None: return override, np.nan, np.nan, None
    if times is None: return 'unknown', np.nan, np.nan, None
    good = pd.Series(times).dropna().sort_values()
    diffs = good.diff().dropna()
    diffs = diffs[diffs > pd.Timedelta(0)]
    if diffs.empty: return 'unknown', np.nan, np.nan, None
    step = diffs.mode().iloc[0]
    consistency = float((diffs == step).mean())
    minutes = step.total_seconds()/60
    if minutes < 60: text = f'{minutes:g} min'
    elif minutes < 1440: text = f'{minutes/60:g} hour'
    else: text = f'{minutes/1440:g} day'
    return text, float(minutes), consistency, step

def choose_period(name, interval_minutes, train_len):
    if name in SEASONAL_PERIOD_OVERRIDE:
        return int(SEASONAL_PERIOD_OVERRIDE[name]), 'manual'
    if not np.isfinite(interval_minutes) or interval_minutes <= 0: return None, 'unknown frequency'
    # Hydrological 'daily' is best characterized annually when 2+ years of TRAIN data exist.
    if name.casefold().startswith('piezo') and np.isclose(interval_minutes, 1440) and train_len >= 2*365:
        return 365, 'annual (daily observations)'
    # For hourly and subhourly series choose daily cycle if feasible.
    daily = 1440 / interval_minutes
    if abs(daily-round(daily)) < 1e-6 and 2 <= round(daily) <= train_len//2:
        return int(round(daily)), 'daily'
    weekly = 10080 / interval_minutes
    if abs(weekly-round(weekly)) < 1e-6 and 2 <= round(weekly) <= train_len//2:
        return int(round(weekly)), 'weekly'
    return None, 'no supported cycle'

def clipped_strength(residual, competing):
    den = float(np.var(competing))
    return float(np.clip(1 - np.var(residual)/den, 0, 1)) if den > 0 else np.nan

def analyze_file(name, path, family, target):
    raw = pd.read_csv(path)
    date_col = resolve_date_column(raw)
    if target not in raw.columns:
        raise ValueError(f'{name}: expected target {target!r} not found. Available: {list(raw.columns)}')
    df = raw.copy()
    if date_col:
        parsed = pd.to_datetime(df[date_col], errors='coerce')
        if parsed.isna().any():
            warnings.warn(f'{name}: {parsed.isna().sum()} unparseable timestamps; retained for diagnostic purposes.')
        df['_parsed_date_'] = parsed
        df = df.sort_values('_parsed_date_', kind='stable').reset_index(drop=True)
        dates = df['_parsed_date_']
    else: dates = None
    numeric_cols = [c for c in raw.columns if c != date_col and pd.api.types.is_numeric_dtype(raw[c])]
    nonnumeric_cols = [c for c in raw.columns if c != date_col and c not in numeric_cols]
    n = len(df)
    freq, minutes, regularity, step = find_frequency(dates, FREQUENCY_OVERRIDE.get(name))
    missing_target = int(pd.to_numeric(df[target], errors='coerce').isna().sum())
    missing_any = int(df[numeric_cols].isna().any(axis=1).sum()) if numeric_cols else np.nan
    duplicates = int(dates.duplicated().sum()) if dates is not None else np.nan
    train = df.iloc[:int(n*TRAIN_RATIO)].copy()
    y = pd.to_numeric(train[target], errors='coerce')
    period, rationale = choose_period(name, minutes, len(train))
    notes = []
    if not date_col: notes.append('date column absent')
    if nonnumeric_cols: notes.append('nonnumeric other columns: '+', '.join(nonnumeric_cols))
    if missing_target: notes.append(f'{missing_target} missing target values')
    if dates is not None and dates.isna().any(): notes.append('unparseable dates')
    if np.isfinite(regularity) and regularity < .98: notes.append('irregular timestamps/gaps')
    if dates is not None and duplicates: notes.append(f'{duplicates} duplicated timestamps')
    season = trend = noise = adf_p = np.nan
    usable = y.dropna().to_numpy(dtype=float)
    if period is None or period < 2: notes.append('no defensible STL seasonal period')
    elif len(usable) < 2*period: notes.append('insufficient train points for 2 seasonal cycles')
    elif dates is not None and (not np.isfinite(regularity) or regularity < .98 or duplicates or dates.isna().any()):
        notes.append('STL skipped: irregular/duplicate/unparseable dates')
    elif len(usable) != len(y): notes.append('STL skipped: missing training target')
    elif np.std(usable) <= 0: notes.append('STL skipped: constant target')
    else:
        # Take last contiguous part of TRAIN. Full observed feature count / timespan still use entire CSV.
        block = usable[-max(MAX_STL_TRAIN_POINTS, 2*period):]
        decomposition = STL(block, period=period, seasonal=13, robust=True).fit()
        season = clipped_strength(decomposition.resid, decomposition.seasonal+decomposition.resid)
        trend = clipped_strength(decomposition.resid, decomposition.trend+decomposition.resid)
        noise = float(np.var(decomposition.resid)/np.var(block)) if np.var(block)>0 else np.nan
    if len(usable) >= 50 and np.std(usable)>0 and (dates is None or (np.isfinite(regularity) and regularity >= .98 and not duplicates)):
        try:
            adf_p = float(adfuller(usable[-MAX_ADF_TRAIN_POINTS:], maxlag=12, autolag='AIC')[1])
        except Exception as exc: notes.append('ADF error: '+str(exc)[:80])
    else: notes.append('ADF skipped: missing/short/irregular or constant target')
    result = dict(dataset=name, family=family, source='measured CSV', path=str(path),
        frequency=freq, modal_interval_minutes=minutes, frequency_consistency=regularity,
        variables_M=len(numeric_cols), timestamps_T=n, target=target,
        start_date=str(dates.min())[:19] if dates is not None else '',
        end_date=str(dates.max())[:19] if dates is not None else '',
        missing_target=missing_target, rows_with_missing_numeric=missing_any,
        duplicate_timestamps=duplicates, train_timesteps=len(train),
        STL_period=period, STL_cycle=rationale, seasonality_strength=season, trend_strength=trend,
        noise_ratio=noise, ADF_p=adf_p,
        ADF_reject_5pct='Yes' if np.isfinite(adf_p) and adf_p < .05 else ('No' if np.isfinite(adf_p) else 'n/a'),
        seasonality_label=strength_label(season), trend_label=strength_label(trend), noise_label=noise_label(noise),
        notes='; '.join(notes))
    return result

rows=[]
for name,path in piezo_files.items():
    try: rows.append(analyze_file(name,path,'BRGM','level'))
    except Exception as e: print('ERROR', name, repr(e)); raise
for name,path in public_files.items():
    target=TARGETS.get(name)
    if name=='Electricity':
        print('Electricity has no universally defined one target in the 321-series benchmark. '
              'Table structural properties are calculated with its first numeric series; '
              'change TARGETS["Electricity"] to the actual target used in your CF experiment.')
        raw=pd.read_csv(path,nrows=1)
        date_col=resolve_date_column(raw)
        cols=[c for c in raw.columns if c != date_col]
        target=TARGETS.get(name,cols[0])
    if name=='Weather' and target not in pd.read_csv(path,nrows=1).columns:
        raise ValueError(f'Weather: target {target!r} missing; inspect CSV and update TARGETS')
    rows.append(analyze_file(name,path,'Public',target))
measured=pd.DataFrame(rows)
pd.set_option('display.max_columns',40)
display(measured[['dataset','frequency','variables_M','timestamps_T','target',
                 'STL_period','seasonality_strength','trend_strength','noise_ratio','ADF_p','notes']])
measured.to_csv(OUTPUT_ROOT/'dataset_properties_measured.csv',index=False)
print('Saved:', OUTPUT_ROOT/'dataset_properties_measured.csv')


/tmp/ipykernel_641/3111557771.py:90: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_p = float(adfuller(usable[-MAX_ADF_TRAIN_POINTS:], maxlag=12, autolag='AIC')[1])
/tmp/ipykernel_641/3111557771.py:90: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  adf_p = float(adfuller(usable[-MAX_ADF_TRAIN_POINTS:], maxlag=12, autolag='AIC')[1])
/tmp/ipykernel_641/3111557771.py:90: F

,dataset,frequency,variables_M,timestamps_T,target,STL_period,seasonality_strength,trend_strength,noise_ratio,ADF_p,notes
0,piezo1,1 day,4,10865,level,365,0.542345,0.208387,0.410856,7.785261e-05,
1,piezo2,1 day,4,18074,level,365,0.000000,0.998014,0.001989,2.415466e-01,
2,piezo3,1 day,4,10874,level,365,0.751364,0.347749,0.225227,1.386225e-04,
3,piezo4,1 day,4,10766,level,365,0.465908,0.980380,0.019378,8.063467e-01,
4,piezo5,1 day,4,9129,level,365,0.527655,0.531169,0.302434,3.408337e-03,
5,piezo6,1 day,4,21036,level,365,0.297196,0.992883,0.007115,8.932149e-01,
6,piezo7,1 day,4,18082,level,365,0.378580,0.962499,0.036708,3.423550e-02,
7,piezo8,1 day,4,10796,level,365,0.754241,0.540132,0.190204,4.004340e-04,
8,piezo9,1 day,4,20727,level,365,0.006512,0.995637,0.004365,9.483672e-01,
9,piezo11,1 day,4,21386,level,365,0.144719,0.996593,0.003403,9.784657e-01,


Saved: /content/drive/MyDrive/EDBT_dataset_characterization/dataset_properties_measured.csv


## Missing public dataset values: explicitly attributed screenshot references

Public rows are filled from **your uploaded image** only when their CSV was not supplied. They have no numerical STL or ADF estimates. The screenshot gives Weather frequency `1-hour`; do not use that number in the final paper without checking the downloaded `weather.csv` timestamps. Similarly, “Low stationarity” in the screenshot is **not an ADF result** and must not be silently mapped to an ADF result.

In [25]:
REFERENCE_PUBLIC = {
  'ETTh1':      dict(frequency='1 hour',variables_M=7,timestamps_T=17420,seasonality_label='High',trend_label='High',noise_label='Low–medium',stationarity_source='Low'),
  'ETTh2':      dict(frequency='1 hour',variables_M=7,timestamps_T=17420,seasonality_label='Med.–high',trend_label='High',noise_label='Low',stationarity_source='Low'),
  'Weather':    dict(frequency='1 hour [VERIFY]',variables_M=21,timestamps_T=52696,seasonality_label='High',trend_label='Med.–high',noise_label='Medium',stationarity_source='Low'),
  'Electricity':dict(frequency='1 hour',variables_M=321,timestamps_T=26304,seasonality_label='Med.–high',trend_label='High',noise_label='Medium',stationarity_source='Low'),
}
all_rows = list(rows)
for name in PUBLIC_NAMES:
    if name not in public_files:
        ref=REFERENCE_PUBLIC[name]
        all_rows.append(dict(dataset=name,family='Public',source='user-provided screenshot, NOT measured',
              path='',target=TARGETS.get(name,'not designated'),frequency=ref['frequency'],
              variables_M=ref['variables_M'],timestamps_T=ref['timestamps_T'],
              seasonality_strength=np.nan,trend_strength=np.nan,noise_ratio=np.nan,ADF_p=np.nan,
              ADF_reject_5pct='not tested', seasonality_label=ref['seasonality_label'],
              trend_label=ref['trend_label'],noise_label=ref['noise_label'],
              stationarity_source=ref['stationarity_source'],STL_period=np.nan,
              notes='Qualitative categories are transcribed from screenshot; different/unknown definitions'))
combined=pd.DataFrame(all_rows)
combined['sort_index']=combined['dataset'].map({s:i for i,s in enumerate(PIEZO_NAMES+PUBLIC_NAMES)})
combined=combined.sort_values('sort_index').drop(columns='sort_index').reset_index(drop=True)
combined.to_csv(OUTPUT_ROOT/'dataset_properties_with_screenshot_references.csv',index=False)
print('BRGM measured:',sum(combined.family=='BRGM'),'Public measured:',len(public_files),
      'Public screenshot placeholders:',4-len(public_files))
display(combined[['dataset','source','frequency','variables_M','timestamps_T','seasonality_label','trend_label','noise_label','ADF_reject_5pct']])


BRGM measured: 17 Public measured: 0 Public screenshot placeholders: 4


,dataset,source,frequency,variables_M,timestamps_T,seasonality_label,trend_label,noise_label,ADF_reject_5pct
0,piezo1,measured CSV,1 day,4,10865,Med.-high,Medium,High,Yes
1,piezo2,measured CSV,1 day,4,18074,Low,High,Low,No
2,piezo3,measured CSV,1 day,4,10874,High,Medium,Medium,Yes
3,piezo4,measured CSV,1 day,4,10766,Med.-high,High,Low,No
4,piezo5,measured CSV,1 day,4,9129,Med.-high,Med.-high,High,Yes
5,piezo6,measured CSV,1 day,4,21036,Medium,High,Low,No
6,piezo7,measured CSV,1 day,4,18082,Medium,High,Low,Yes
7,piezo8,measured CSV,1 day,4,10796,High,Med.-high,Medium,Yes
8,piezo9,measured CSV,1 day,4,20727,Low,High,Low,No
9,piezo11,measured CSV,1 day,4,21386,Low,High,Low,No


## Export LaTeX tables and a provenance-aware methods note

`main_dataset_table.tex` = compact table suitable for the **main EDBT paper** (17 BRGM datasets pooled into a single descriptive group, plus four individual public datasets). `full_dataset_table.tex` = all 21 rows, useful in appendix/supplement. The compact table reports **ranges for BRGM** where datasets differ; seasonality/trend/noise are shown as medians of the numeric scores, rather than misleadingly calling 17 different time series all “high”. The detailed table reports the custom categorical labels and target-specific ADF outcome. Screenshot-only entries are footnoted as reference values.

**Do not report the generated table as final if data-quality warnings, missing piezometers, or unverified Weather frequency remain.**

In [26]:
def latex_escape(value):
    value=str(value)
    return (value.replace('\\','\\textbackslash{}').replace('&',r'\&').replace('%',r'\%')
            .replace('_',r'\_').replace('#',r'\#'))

def nice_range(vals, formatter=lambda x: f'{int(x):,}'):
    vals=pd.to_numeric(pd.Series(vals),errors='coerce').dropna()
    if vals.empty: return '--'
    lo,hi=vals.min(),vals.max()
    return formatter(lo) if lo==hi else formatter(lo)+'--'+formatter(hi)

def brgm_summary(df):
    b=df[df.family=='BRGM']
    if b.empty: raise ValueError('BRGM data missing')
    freqs=sorted(b.frequency.dropna().astype(str).unique())
    freq=freqs[0] if len(freqs)==1 else 'varies'
    def median_score(field):
        v=pd.to_numeric(b[field],errors='coerce').dropna()
        return f'{v.median():.2f}' if len(v) else '--'
    return dict(dataset=f'BRGM piezometers ($n={len(b)}$)',frequency=freq,
        variables_M=nice_range(b.variables_M),timestamps_T=nice_range(b.timestamps_T),
        seasonal=median_score('seasonality_strength'),trend=median_score('trend_strength'),
        noise=median_score('noise_ratio'))

mainlines=[r'\begin{table*}[t]',r'\centering',r'\caption{Characteristics of the multivariate forecasting datasets. BRGM entries summarize the 17 individual piezometric series; other datasets are reported individually. Numeric seasonality and trend strengths and noise ratio are training-target STL summaries; a dash indicates that only screenshot-reported qualitative information was available.}',
 r'\label{tab:dataset_characteristics}',r'\footnotesize',r'\setlength{\tabcolsep}{4pt}',
 r'\begin{tabular}{@{}lccccccc@{}}',r'\toprule',
 r'Dataset & Frequency & Variables & Time steps & Seasonality & Trend & Noise & Source \\',
 r'\midrule']
b=brgm_summary(combined)
mainlines.append(f"{b['dataset']} & {latex_escape(b['frequency'])} & {b['variables_M']} & {b['timestamps_T']} & {b['seasonal']} & {b['trend']} & {b['noise']} & measured \\\\")
mainlines.append(r'\midrule')
for name in PUBLIC_NAMES:
    r=combined[combined.dataset==name].iloc[0]
    was_measured=r.source=='measured CSV'
    def val(key,fmt='.2f'):
        x=pd.to_numeric(pd.Series([r.get(key)]),errors='coerce').iloc[0]
        return format(x,fmt) if np.isfinite(x) else '--'
    season=val('seasonality_strength') if was_measured else latex_escape(r.seasonality_label)+'$^{\dagger}$'
    trend=val('trend_strength') if was_measured else latex_escape(r.trend_label)+'$^{\dagger}$'
    noise=val('noise_ratio') if was_measured else latex_escape(r.noise_label)+'$^{\dagger}$'
    source='measured' if was_measured else 'ref.$^{\dagger}$'
    mainlines.append(f"{latex_escape(name)} & {latex_escape(r.frequency)} & {int(r.variables_M):,} & {int(r.timestamps_T):,} & {season} & {trend} & {noise} & {source} \\\\")
mainlines += [r'\bottomrule',r'\end{tabular}',
 r'\vspace{1mm}',r'\parbox{0.98\textwidth}{\scriptsize \textit{Notes:} Variables count excludes the timestamp column. The BRGM entry reports the range of feature counts and series lengths and the median of training-target STL scores. $^{\dagger}$Reference-only descriptive labels and counts transcribed from the supplied comparison graphic; source-defined thresholds are unavailable. Confirm Weather timestamp frequency against the actual CSV before publication. Measured values and reference labels are not directly comparable.}',r'\end{table*}']
maintex='\n'.join(mainlines)+'\n'
(OUTPUT_ROOT/'main_dataset_table.tex').write_text(maintex,encoding='utf8')

fulllines=[r'\begin{table*}[t]',r'\centering',r'\caption{Individual time-series dataset characteristics. BRGM statistics are computed from the target variable on the first 50\% of each series; reference-only public dataset labels are transcribed from the provided graphic.}',
 r'\label{tab:dataset_characteristics_full}',r'\scriptsize',r'\setlength{\tabcolsep}{3pt}',
 r'\begin{tabular}{@{}llrrccccc@{}}',r'\toprule',
 r'Dataset & Sampling & $M$ & $T$ & Seasonality & Trend & Noise & ADF $p$ & Provenance \\',r'\midrule']
for _,r in combined.iterrows():
    def categorical(field): return latex_escape(r[field])
    adfp=pd.to_numeric(pd.Series([r.get('ADF_p')]),errors='coerce').iloc[0]
    adf=f'{adfp:.3g}' if np.isfinite(adfp) else '--'
    prov='CSV' if r.source=='measured CSV' else 'graphic$^{\dagger}$'
    fulllines.append(f"{latex_escape(r.dataset)} & {latex_escape(r.frequency)} & {int(r.variables_M):,} & {int(r.timestamps_T):,} & {categorical('seasonality_label')} & {categorical('trend_label')} & {categorical('noise_label')} & {adf} & {prov} \\\\")
    if r.dataset==PIEZO_NAMES[-1]: fulllines.append(r'\midrule')
fulllines += [r'\bottomrule',r'\end{tabular}',r'\vspace{1mm}',
 r'\parbox{0.98\textwidth}{\scriptsize \textit{Notes:} $M$ excludes dates; $T$ is the number of CSV rows. For measured rows, High/Med.-high/Medium/Low seasonality and trend correspond to STL strengths $[0.70,1]$, $[0.40,0.70)$, $[0.20,0.40)$, and $[0,0.20)$; noise Low/Medium/High corresponds to STL residual-to-observed variance ratio below 0.10, 0.10--0.30, and above 0.30. ADF is the training-target augmented Dickey--Fuller $p$-value, not the graphic\'s stationarity label. $^{\dagger}$ Graphic labels use unknown original thresholds and should not be used for statistical ranking against measured entries.}',r'\end{table*}']
fulltex='\n'.join(fulllines)+'\n'
(OUTPUT_ROOT/'full_dataset_table.tex').write_text(fulltex,encoding='utf8')

methods_note=(
  'Training-target properties: STL(period chosen by recorded sampling interval; '
  'annual for daily BRGM when training has >= 730 samples, else daily/weekly where possible), '
  'last <= 6000 training observations (at least two cycles), robust=True, seasonal=13. '
  'Seasonality strength=max(0,1-Var(R)/Var(S+R)); trend strength=max(0,1-Var(R)/Var(T+R)); '
  'noise proxy=Var(R)/Var(observed). ADF p from last <= 5000 training observations '
  '(maxlag=12, autolag=AIC). Full-series M,T and frequency inferred from CSV; '
  'only complete, adequately regular series are decomposed. '
  'Missing public CSVs are screenshot references only; Weather frequency requires checking.'
)
(OUTPUT_ROOT/'dataset_methods_note.txt').write_text(methods_note,encoding='utf8')
print('Created:',OUTPUT_ROOT/'main_dataset_table.tex')
print('Created:',OUTPUT_ROOT/'full_dataset_table.tex')
print('Created:',OUTPUT_ROOT/'dataset_methods_note.txt')
print('\n--- MAIN TABLE PREVIEW ---\n',maintex[:3400])

Created: /content/drive/MyDrive/EDBT_dataset_characterization/main_dataset_table.tex
Created: /content/drive/MyDrive/EDBT_dataset_characterization/full_dataset_table.tex
Created: /content/drive/MyDrive/EDBT_dataset_characterization/dataset_methods_note.txt

--- MAIN TABLE PREVIEW ---
 \begin{table*}[t]
\centering
\caption{Characteristics of the multivariate forecasting datasets. BRGM entries summarize the 17 individual piezometric series; other datasets are reported individually. Numeric seasonality and trend strengths and noise ratio are training-target STL summaries; a dash indicates that only screenshot-reported qualitative information was available.}
\label{tab:dataset_characteristics}
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{@{}lccccccc@{}}
\toprule
Dataset & Frequency & Variables & Time steps & Seasonality & Trend & Noise & Source \\
\midrule
BRGM piezometers ($n=17$) & 1 day & 4 & 7,530--21,386 & 0.35 & 0.90 & 0.09 & measured \\
\midrule
ETTh1 & 1 hour & 7 & 17,

<>:39: SyntaxWarning: invalid escape sequence '\d'
<>:40: SyntaxWarning: invalid escape sequence '\d'
<>:41: SyntaxWarning: invalid escape sequence '\d'
<>:42: SyntaxWarning: invalid escape sequence '\d'
<>:57: SyntaxWarning: invalid escape sequence '\d'
<>:39: SyntaxWarning: invalid escape sequence '\d'
<>:40: SyntaxWarning: invalid escape sequence '\d'
<>:41: SyntaxWarning: invalid escape sequence '\d'
<>:42: SyntaxWarning: invalid escape sequence '\d'
<>:57: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_641/1392411962.py:39: SyntaxWarning: invalid escape sequence '\d'
  season=val('seasonality_strength') if was_measured else latex_escape(r.seasonality_label)+'$^{\dagger}$'
/tmp/ipykernel_641/1392411962.py:40: SyntaxWarning: invalid escape sequence '\d'
  trend=val('trend_strength') if was_measured else latex_escape(r.trend_label)+'$^{\dagger}$'
/tmp/ipykernel_641/1392411962.py:41: SyntaxWarning: invalid escape sequence '\d'
  noise=val('noise_ratio') if was_measured els

## Before pasting tables into Overleaf

1. Review `dataset_properties_measured.csv` for timestamp irregularities and `notes`. Verify `17/17` BRGM files were found and counted; your CF notebook omits `piezo10` and includes `piezo18`.
2. Prefer putting ETTh1, ETTh2, Weather, Electricity CSVs under `PUBLIC_ROOT` and **rerun everything**: it replaces graphic-derived categorical labels with measured numeric metrics for all 21 datasets. The external datasets may need a different target definition (especially Electricity): verify that `TARGETS` matches your actual forecasting task.
3. Include `\usepackage{booktabs}` in your LaTeX preamble, and put `\input{main_dataset_table.tex}` in the main text. The 21-row table is optional in an appendix/supplement (`\input{full_dataset_table.tex}`).
4. If your EDBT format does not allow wide floats, keep the aggregate main table and place the 21-row details in the supplementary material. If the main paper only has space for one table, **use the five-row version** and say that BRGM has 17 separate sites.
5. Check original dataset documentation and cite dataset sources in the paper; this notebook creates descriptive values but does not establish the provenance of the public datasets or BRGM licensing.